In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ==========================================
# FAST + HIGH ACCURACY (EFFICIENTNET-B0)
# ==========================================

import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ======================
# PATHS
# ======================
TRAIN_DIR = "/kaggle/input/datasets/mstjebashazida/affectnet/archive (3)/Train"
TEST_DIR = "/kaggle/input/datasets/mstjebashazida/affectnet/archive (3)/Test"

# ======================
# TRANSFORMS (FAST + GOOD)
# ======================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # 🔥 reduced from 300
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ======================
# DATASETS
# ======================
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

print("Classes:", train_dataset.classes)

# ======================
# DATALOADER (FAST)
# ======================
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=2, pin_memory=True)

# ======================
# DEVICE
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# MODEL (FAST + POWERFUL)
# ======================
model = models.efficientnet_b0(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(train_dataset.classes))

model = model.to(device)

# ======================
# LOSS + OPTIMIZER
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# ======================
# TRAIN LOOP
# ======================
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"\n🚀 Starting Epoch {epoch+1}")

    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # 🔥 progress print (so you know it's running)
        if i % 100 == 0:
            print(f"Batch {i}/{len(train_loader)}")

    train_acc = 100 * correct / total

    # ======================
    # VALIDATION
    # ======================
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("✅ Training Complete")

Classes: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 20.5M/20.5M [00:00<00:00, 147MB/s]



🚀 Starting Epoch 1
Batch 0/504
Batch 100/504
Batch 200/504
Batch 300/504
Batch 400/504
Batch 500/504
Epoch 1 | Loss: 1.0774 | Train Acc: 59.29% | Val Acc: 64.62%

🚀 Starting Epoch 2
Batch 0/504
Batch 100/504
Batch 200/504
Batch 300/504
Batch 400/504
Batch 500/504
Epoch 2 | Loss: 0.8428 | Train Acc: 69.18% | Val Acc: 66.88%

🚀 Starting Epoch 3
Batch 0/504
Batch 100/504
Batch 200/504
Batch 300/504
Batch 400/504
Batch 500/504
Epoch 3 | Loss: 0.7545 | Train Acc: 72.67% | Val Acc: 69.18%

🚀 Starting Epoch 4
Batch 0/504
Batch 100/504
Batch 200/504
Batch 300/504
Batch 400/504
Batch 500/504
Epoch 4 | Loss: 0.6824 | Train Acc: 74.85% | Val Acc: 67.39%

🚀 Starting Epoch 5
Batch 0/504
Batch 100/504
Batch 200/504
Batch 300/504
Batch 400/504
Batch 500/504
Epoch 5 | Loss: 0.6119 | Train Acc: 77.23% | Val Acc: 70.19%

🚀 Starting Epoch 6
Batch 0/504
Batch 100/504
Batch 200/504
Batch 300/504
Batch 400/504
Batch 500/504
Epoch 6 | Loss: 0.5568 | Train Acc: 79.44% | Val Acc: 70.01%

🚀 Starting Epoch 7
Ba